# GE 对接 PyTorch — TorchAir 图模式

PyTorch 用户在昇腾上有两种运行方式：**单算子（Eager）模式**与**图模式**。Eager 模式即时下发、调试直观，但缺乏整图视角；图模式则把一段 `forward` 捕获成一张计算图，交由 GE 做整图优化与下沉执行，从而获得更优性能。TorchAir 就是 PyTorch 接入 GE 图模式的官方入口。

本节带你看清「PyTorch → TorchAir → AscendIR → GE」这条链路，掌握图模式的启用与配置，并明确它的使用边界。

本节学习大纲如下：

- TorchAir 是什么：在昇腾全栈中的定位
- 架构链路：Dynamo/FX 图捕获 → AscendIR → GE 编译执行
- 图模式启用：`torch.compile` + NPU backend
- 编译配置与缓存（CompilerConfig）
- GE 图下沉与 Host 调度开销
- 回退（fallback）到 Eager 的机制
- 使用边界：动态 shape / 控制流 / 不支持算子
- 接入 checklist

> 说明：本节的动手代码按 CANN 9.0 及以上的配套 `torch`/`torch_npu` 环境编写。代码单元会真实完成 Eager 对拍、TorchAir 图编译和重复执行计时；版本相关的高级配置仍应以当前环境的 `torchair` 文档为准。

## 1. TorchAir 是什么

TorchAir（Torch Ascend Intermediate Representation）是 Ascend Extension for PyTorch（`torch_npu`）中支持图模式能力的扩展库。它对接 PyTorch 的 Dynamo 特性，将 PyTorch 捕获到的 FX 图转换为 **AscendIR** 表达的计算图，再交由 **GE** 编译优化并下沉到昇腾硬件执行。

回顾第一章中 GE 的「框架驱动路径」：GE 位于昇腾全栈中间层，向上对接主流框架，向下通过 Runtime 驱动硬件。对 PyTorch 而言，这条「向上对接」的桥梁就是 TorchAir。

| 角色 | 组件 | 职责 |
| --- | --- | --- |
| 前端框架 | PyTorch | 定义模型 `nn.Module`、`forward` 计算逻辑 |
| 图捕获 | TorchDynamo / FX | 在 `torch.compile` 触发时，把 Python 字节码捕获为 FX Graph |
| 桥接转换 | **TorchAir** | 将 FX Graph（aten IR）转换为 AscendIR 计算图，并提供 NPU backend |
| 图编译执行 | **GE** | 接收 AscendIR，做整图优化、内存/流编排，编译并下沉执行 |
| 运行时/硬件 | Runtime / AI Core / AI CPU | 实际执行算子计算 |

> TorchAir 名字里的 "Air" 即取自 AscendIR（AIR）。它负责将 PyTorch 的 FX Graph 转换为 GE 使用的 AscendIR 计算图。

## 2. 架构链路：从 forward 到昇腾硬件

TorchAir 图模式的端到端链路如下：

<p align="left"><img src="./images/torchair_pipeline.svg" alt="PyTorch 图模式 TorchAir 链路" width="100%"></p>

链路要点：

1. **图捕获（Dynamo/FX）**：`torch.compile` 在首次执行被装饰的函数时，由 TorchDynamo 分析 Python 字节码，把可追踪的张量计算捕获为 FX Graph，FX 节点是 **aten 级算子**（如 `aten.add`、`aten.matmul`）。
2. **aten IR → AscendIR**：TorchAir 提供的 NPU backend 接收 FX Graph，把 aten 算子逐个映射为对应的 GE/AscendIR 算子，组装成一张 AscendIR 计算图。
3. **GE 编译执行**：AscendIR 图进入 GE（在线路径，通过 GeSession 风格的编译执行），完成图优化、算子融合、内存复用、流分配，并以**模型下沉**方式执行（参见第一章「模型下沉调度」）。
4. **回到 PyTorch**：执行结果以 PyTorch Tensor 返回，对上层业务代码透明。

> 与离线 ATC 路径相比，TorchAir 走的是**在线编译执行**：图在运行时由框架实时构建、编译并执行，无需手工导出 ONNX/OM。

## 3. 图模式启用：torch.compile + NPU backend

启用图模式的核心，是把 TorchAir 提供的昇腾 backend 传给 `torch.compile`。下面先给出接口范式，随后用一个小型 MLP 在真实 NPU 上完成同样的流程：

```python
# 1) 导入顺序很重要：先 torch，再 torch_npu，再 torchair
import torch
import torch_npu
import torchair

# 2) 定义模型
class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x, y):
        return torch.add(x, y)

model = Model().npu()        # 模型/数据需放到 NPU 设备

# 3) 从 TorchAir 获取 NPU backend
config = torchair.CompilerConfig()                       # 编译配置（见第 4 节）
config.mode = "max-autotune"                 # GE/AscendIR 图模式
npu_backend = torchair.get_npu_backend(compiler_config=config)

# 4) 用 NPU backend 编译模型，使能图模式
opt_model = torch.compile(model, backend=npu_backend)

# 5) 像普通模型一样调用，首次调用触发图捕获+编译，后续命中缓存
x = torch.randn(2, 2).npu()
y = torch.randn(2, 2).npu()
out = opt_model(x, y)
```

关键点对照表：

| 步骤 | 关键动作 | 说明 |
| --- | --- | --- |
| 导入 | `import torch_npu` → `import torchair` | torch_npu 注册 NPU 设备/算子，torchair 提供图模式 backend |
| 设备 | `.npu()` | 模型与输入张量都需在 NPU 上 |
| backend | `torchair.get_npu_backend(...)` | 返回可传给 `torch.compile` 的昇腾后端 |
| 编译 | `torch.compile(model, backend=...)` | 触发 Dynamo 捕获 + TorchAir 转图 + GE 编译 |
| 首次执行 | `opt_model(x, y)` | **首次较慢**（含捕获/编译），后续命中编译缓存 |

> 提示：`torch.compile` 是**惰性编译**：定义 `opt_model` 时并不会立刻编译，真正的捕获与编译发生在**第一次实际调用**时。

### 3.1 动手实践：Eager 与 TorchAir 图模式对拍

下面使用一个最小 MLP：先构造可复现的输入和 CPU 参考结果，再在 NPU 上分别执行 Eager 与 TorchAir 图模式，最后做数值对拍并观察首次编译与稳定执行的耗时。

运行前请先加载 CANN 环境（例如 `source /usr/local/Ascend/ascend-toolkit/set_env.sh`），并确认 `torch_npu` 与当前 PyTorch 版本匹配。本单元需要真实的 NPU 设备；没有 NPU 时会直接报错，不提供 CPU 降级路径，避免把 Host 模拟误当成图模式结果。这里显式设置 `fullgraph=True`，用于验证整段逻辑能否形成一张图；遇到 graph break 时会直接报错，而不是自动回退到 Eager。

In [ ]:
# === 真机运行：PyTorch Eager -> TorchAir/GE 图模式 -> 数值对拍与计时 ===
import time

# 导入顺序：torch -> torch_npu -> torchair。torch_npu 负责注册 NPU，
# torchair 提供把 FX 图交给 GE 的 backend。
import torch
import torch_npu
import torchair

if not hasattr(torch, "npu") or not torch.npu.is_available():
    raise RuntimeError(
        "未检测到可用 NPU。请先 source CANN 的 set_env.sh，并在昇腾设备上运行本单元。"
    )

print("PyTorch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("NPU:", torch.npu.get_device_name(0))
torch.manual_seed(2026)

class TinyMLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.weight1 = torch.nn.Parameter(torch.randn(8, 16))
        self.bias1 = torch.nn.Parameter(torch.randn(16))
        self.weight2 = torch.nn.Parameter(torch.randn(16, 4))
        self.bias2 = torch.nn.Parameter(torch.randn(4))

    def forward(self, x):
        hidden = torch.mm(x, self.weight1) + self.bias1
        hidden = torch.relu(hidden)
        return torch.mm(hidden, self.weight2) + self.bias2

# 先在 CPU 上固定权重和输入，再复制到 NPU，保证 Eager/图模式使用同一组数据。
cpu_model = TinyMLP().eval()
npu_model = TinyMLP().eval()
npu_model.load_state_dict(cpu_model.state_dict())
npu_model = npu_model.npu()
x_cpu = torch.randn(4, 8)
x_npu = x_cpu.npu()

with torch.no_grad():
    cpu_expected = cpu_model(x_cpu)
    eager_output = npu_model(x_npu)
    torch.npu.synchronize()
    eager_output_cpu = eager_output.cpu().clone()
    torch.testing.assert_close(
        eager_output_cpu, cpu_expected, rtol=1e-4, atol=1e-4
    )

# 明确选择 GE 的 max-autotune 路径，把 FX 图转换为 AscendIR 后交给 GE 编译。
config = torchair.CompilerConfig()
config.mode = "max-autotune"
npu_backend = torchair.get_npu_backend(compiler_config=config)
compiled_model = torch.compile(
    npu_model,
    backend=npu_backend,
    fullgraph=True,
    dynamic=False,
)

def run_and_time(fn, *args):
    started = time.perf_counter()
    result = fn(*args)
    # NPU 算子通常异步下发；同步后再读取 Host 时钟。
    torch.npu.synchronize()
    return (time.perf_counter() - started) * 1e3, result

with torch.no_grad():
    eager_ms = [run_and_time(npu_model, x_npu)[0] for _ in range(5)]

with torch.no_grad():
    first_ms, graph_output = run_and_time(compiled_model, x_npu)
    graph_output_cpu = graph_output.float().cpu().clone()
    torch.testing.assert_close(
        graph_output_cpu, cpu_expected, rtol=1e-4, atol=1e-4
    )
    torch.testing.assert_close(
        graph_output_cpu, eager_output_cpu, rtol=1e-4, atol=1e-4
    )

    graph_repeat_ms = []
    for _ in range(5):
        elapsed_ms, graph_output = run_and_time(compiled_model, x_npu)
        graph_repeat_ms.append(elapsed_ms)
        torch.testing.assert_close(
            graph_output.float().cpu(), cpu_expected, rtol=1e-4, atol=1e-4
        )

mode_value = getattr(getattr(config, "mode", None), "value", "unknown")
print("CompilerConfig.mode:", mode_value)
print("输出 shape:", tuple(graph_output_cpu.shape))
print("Eager 5 次执行平均: {:.3f} ms".format(sum(eager_ms) / len(eager_ms)))
print("首次调用（含 Dynamo/GE 编译）: {:.3f} ms".format(first_ms))
print("图模式后续 5 次执行平均: {:.3f} ms".format(
    sum(graph_repeat_ms) / len(graph_repeat_ms)
))
print("[OK] TorchAir 图模式输出与 CPU/Eager 参考结果一致")

> 结果解读：首次调用包含 Dynamo 捕获、TorchAir 转图和 GE 编译，因此通常明显慢于后续调用。这个 MLP 很小，图模式的稳定态不一定比 Eager 更快；实际服务应使用更大、重复执行次数更多的模型，并结合 msprof 分析 Host 调度与 Device 算子耗时。这里首先验收的是图模式跑通且数值正确。

## 4. 编译配置与缓存（CompilerConfig）

图模式的行为通过 `CompilerConfig` 调整。不同 CANN/torch_npu 版本的属性名称会有差异，使用前请以当前环境的 `torchair` 文档为准。本节只演示基于 GE/AscendIR 的路径，明确使用 `CompilerConfig.mode = "max-autotune"`。`npugraph_ex` 是另一条基于 NPUGraph Capture-Replay 的后端路径，公开入口应写成 `torch.compile(..., backend="npugraph_ex")`；即使某些版本的 `CompilerConfig` 也列出该取值，本节仍将它单独列出，避免与 GE 路径混淆。

| 路径 | 参数入口 | 典型取值 | 说明 |
| --- | --- | --- | --- |
| GE 图模式（本节） | `CompilerConfig.mode` | `"max-autotune"` | FX Graph → AscendIR → GE 编译执行 |
| NPUGraph_EX | `torch.compile(..., backend=...)` | `"npugraph_ex"` | 独立的 Capture-Replay 路径，不能当作 GE 的 `mode` 示例 |
| 旧 ACLGraph 兼容路径（版本相关，逐步弃用） | `CompilerConfig.mode` | `"reduce-overhead"` | 仅在当前环境仍支持时使用，以对应版本文档为准 |
| 动态 shape | `torch.compile(..., dynamic=...)` | `True` / `False` | 允许形状变化时复用图或触发再编译，具体能力取决于版本 |
| 图 dump / 调试 | TorchAir/GE 的版本相关配置或环境变量 | 路径开关 | 导出图和编译信息，便于排障 |
| 缓存 | TorchAir/框架的版本相关缓存配置 | 开 / 关 + 缓存目录 | 复用编译产物，减少重复编译 |
| fallback 策略 | `torch.compile(..., fullgraph=...)` | `False`（允许分段）/ `True`（禁止 graph break） | 选择不支持逻辑的处理方式 |

配置 + 缓存的基本写法：

```python
import torchair

config = torchair.CompilerConfig()
config.mode = "max-autotune"                 # 本节走 GE/AscendIR 图模式
# 下列高级属性的名称随版本变化，实际使用前请查当前版本文档：
# config.debug.graph_dump.type = "txt"   # 导出可读图，辅助排障
# config.export ...                       # 导出/缓存相关

npu_backend = torchair.get_npu_backend(compiler_config=config)
opt_model = torch.compile(model, backend=npu_backend)
```

### 编译缓存

`torch.compile` 以输入张量的关键属性（dtype/shape/设备等）作为缓存键：

- 首次调用某组属性 → 触发捕获 + 编译，开销较大；
- 后续相同属性的调用 → 命中缓存，直接执行已编译图；
- 属性变化（如 shape 改变且未声明动态）→ 重新捕获/编译（即 recompile）。

> 频繁 recompile 是图模式常见性能陷阱：若输入 shape 多变，应显式开启动态 shape，避免每个 shape 都触发一次完整编译。

## 5. 图下沉与 Host 调度开销

图模式相对 Eager 的最大收益，来自减少 Host 与 Device 的调度交互。这正是第一章讲过的 GE「模型下沉调度」在 PyTorch 上的体现。

Eager 模式 vs GE 图模式（模型下沉）调度对比：

```
Eager（逐算子下发）：
  Host:  [下发 add][下发 matmul][下发 relu] ... 每次迭代都重复遍历下发
  Device:        [add]      [matmul]     [relu]
           ↑ Host 频繁与 Device 交互，调度成为瓶颈

GE 图模式（整图下沉）：
  Host:  [下发"模型执行 Task"]            （多次迭代只下发一个 Task）
  Device: [add → matmul → relu 整图一次触发执行]
           ↑ 一次下发触发整图，Host 调度开销大幅降低
```

收益与代价：

| 维度 | Eager | GE 图模式（模型下沉） |
| --- | --- | --- |
| 首次开销 | 无编译 | 有捕获 + 编译头开销 |
| 单步调度开销 | 高（逐算子下发） | 低（整图一次下发） |
| 整图优化 | 无 | 有（融合/内存复用/多流） |
| 适合场景 | 调试、形状高频变化 | 稳定 shape、重复多次执行的推理/训练 |

> 一般来说，模型重复执行次数越多、输入 shape 越稳定，图模式越容易体现优势；调度头开销越小、迭代次数越多，端到端收益越明显。

## 6. 回退（fallback）到 Eager 的机制

以下回退说明针对 `fullgraph=False`（默认）场景：Dynamo 允许把可追踪部分拆成多个子图，不支持的部分在子图之间回到 Eager 执行。第 3.1 节的实践代码显式使用 `fullgraph=True`，该模式禁止 graph break；遇到无法追踪或不支持的逻辑会直接报错，而不是自动 fallback。排障时可以先用 `fullgraph=False` 定位断点，再切回 `fullgraph=True` 验证整图。

常见触发 graph break / 回退的情况：

| 触发点 | 说明 | 影响 |
| --- | --- | --- |
| Python 控制流依赖张量值 | 如 `if tensor.item() > 0:` 这类需要读出张量值的分支 | `fullgraph=False` 时在该处打断成多段图；`fullgraph=True` 时直接报错 |
| 不支持的算子 / 自定义 Python 逻辑 | backend 无对应映射 | `fullgraph=False` 时分段回退 Eager；`fullgraph=True` 时直接报错 |
| `.item()` / `.numpy()` / print 张量 | 触发同步、跳出图 | `fullgraph=False` 时产生 graph break；`fullgraph=True` 时直接报错 |
| 数据依赖的动态 shape（未声明） | 形状由运行时数据决定 | 可能 recompile；`fullgraph=False` 时也可能分段回退，`fullgraph=True` 时直接报错 |

排查建议：

- 多个 graph break 会把一张图切成多段，**优化空间下降、调度开销上升**，性能可能接近甚至劣于 Eager；
- 可借助 Dynamo 的日志/计数（如统计 graph break 次数）定位「在哪里断的」；
- 优化方向：把数据依赖的控制流改写为张量算子、减少 `.item()`/host 同步、对变长输入显式开启动态 shape。

> 回退是功能正确性的兜底，但不保证性能：要拿到完整图模式收益，需尽量减少 graph break；若使用 `fullgraph=True`，则应把 graph break 当作需要修复的错误。

## 7. 使用边界：什么场景适合 / 不适合

| 维度 | 适合图模式 | 需谨慎 / 可能回退 |
| --- | --- | --- |
| **形状** | 静态 shape，或可枚举的少量档位 | 每次 shape 都不同且未声明动态 → 频繁 recompile |
| **控制流** | 固定结构的前向 | 依赖张量**值**的 Python `if/while`（graph break） |
| **算子** | 主流 aten 算子均有映射 | 自定义 Python 逻辑、未支持算子 → 回退 Eager |
| **执行次数** | 重复执行多次（训练 step / 批量推理） | 只跑一两次（编译头开销摊不平） |
| **调试需求** | 性能优先、结构稳定 | 需要逐算子断点调试时 Eager 更直观 |

### 动态 shape 的处理

- 若输入形状会变化（如 NLP 变长序列、视觉动态分辨率），应**显式开启动态 shape**，让 TorchAir/GE 以「分档 / 符号化 shape」方式编译，避免每个具体 shape 都重新编译；
- 完全静态 shape 时，关闭动态 shape 可获得最优执行性能（一次编译、一次下沉）。

> 与第一章呼应：AscendIR 既能表达静态 shape 图，也能表达动态 shape 图，GE 均支持二者的编译与执行；TorchAir 把 PyTorch 的形状信息透传给 GE，由 GE 完成对应编译。

## 8. 接入 checklist 与排障入口

接入 TorchAir 图模式的标准检查清单：

```
[ ] 1. 导入顺序：torch → torch_npu → torchair
[ ] 2. 模型与输入均已 .npu()（在 NPU 设备上）
[ ] 3. 通过 torchair.get_npu_backend(config) 获取 backend
[ ] 4. torch.compile(model, backend=npu_backend)
[ ] 5. 先验证「跑通」（结果与 Eager 一致），再谈「跑快」
[ ] 6. 统计 graph break：若过多，定位数据依赖控制流 / 不支持算子
[ ] 7. shape 多变 → 开启动态 shape，避免频繁 recompile
[ ] 8. 性能不达预期 → 导出图（dump）+ msprof 看算子耗时
```

常见现象与定位入口：

| 现象 | 优先排查 |
| --- | --- |
| 首次很慢、后续正常 | 正常的编译头开销；确认后续命中缓存 |
| 每次都慢 | 频繁 recompile（shape 变化）或大量 graph break |
| 结果与 Eager 不一致 | 缩小到具体子图，对照单算子；先确保数值正确 |
| 报「不支持算子 / 转换失败」 | 该算子回退或缺映射，结合 GE 日志（见 5.4 节） |
| 性能不及预期 | 导出 TorchAir/GE 图 + 用 msprof 分析（见 5.4 节） |

> 图模式跑不通或跑不快时，**与 GE 通用排障一致**：开日志、看错误码、dump 图、用工具分析。具体方法见 **5.4 常见问题定位方法**。

## 课后练习

本节介绍了 PyTorch 经 TorchAir 使用 GE 图模式的架构链路、启用方式与使用边界，请完成以下题目自测。

1. （判断题）TorchAir 是 Ascend Extension for PyTorch（torch_npu）中支持图模式能力的扩展库，负责把 PyTorch 的 FX 图转换为 AscendIR 计算图。

2. （判断题）使用 `torch.compile(model, backend=npu_backend)` 后，模型会在定义 `opt_model` 的那一刻立即完成编译。

3. （单选题）TorchAir 图模式的端到端链路，正确的顺序是？
    A. PyTorch forward → AscendIR → FX Graph → GE 编译执行
    B. PyTorch forward → Dynamo/FX 图（aten IR）→ AscendIR → GE 编译执行
    C. PyTorch forward → OM → ATC → GE 编译执行
    D. PyTorch forward → ONNX → Parser → GE 编译执行

4. （单选题）以下关于「GE 图模式 / 模型下沉」的描述，哪个正确？
    A. 每次迭代都需要 Host 逐算子下发到 Device
    B. 整图一次下发触发执行，显著降低 Host 与 Device 的调度交互
    C. 图模式没有任何首次编译开销
    D. 图模式只能用于推理，不能用于训练

5. （多选题）以下哪些情况可能触发 graph break 或回退 Eager？
    A. 依赖张量值的 Python `if`（如 `if x.item() > 0`）
    B. backend 不支持的算子或自定义 Python 逻辑
    C. 调用 `.item()` / `.numpy()` 导致同步、跳出图
    D. 输入数据依赖的动态 shape 且未声明动态

6. （单选题）某模型每次推理输入 shape 都不同且未开启动态 shape，最可能出现的问题是？
    A. 结果数值错误
    B. 频繁 recompile，性能下降
    C. 无法导入 torchair
    D. 模型无法放到 NPU 设备

7. （多选题）以下关于 TorchAir 接入的最佳实践，哪些正确？
    A. 导入顺序应为 torch → torch_npu → torchair
    B. 模型与输入都需放到 NPU 设备（`.npu()`）
    C. 应先验证「跑通」（与 Eager 结果一致），再优化性能
    D. shape 多变场景应显式开启动态 shape，减少重复编译

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/05.02_answer.txt